# GE Detection Report


In [0]:
from datetime import date
from functools import reduce
import json

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T

dbutils.widgets.text("catalog_name", "hant-catalog")
dbutils.widgets.text("schema_name", "hsl")
dbutils.widgets.text("report_start_date", "")
dbutils.widgets.text("report_end_date", "")
dbutils.widgets.dropdown("persist_report_tables", "true", ["true", "false"])
dbutils.widgets.text("storage_account", "streanmingdatasta")
dbutils.widgets.text("lakehouse_container", "lakehouse")
dbutils.widgets.text("report_base_path", "")

CATALOG_NAME = dbutils.widgets.get("catalog_name")
SCHEMA_NAME = dbutils.widgets.get("schema_name")
REPORT_START_DATE = dbutils.widgets.get("report_start_date").strip() or date.today().isoformat()
REPORT_END_DATE = dbutils.widgets.get("report_end_date").strip() or REPORT_START_DATE
PERSIST_REPORT_TABLES = dbutils.widgets.get("persist_report_tables").lower() == "true"
STORAGE_ACCOUNT = dbutils.widgets.get("storage_account")
LAKEHOUSE_CONTAINER = dbutils.widgets.get("lakehouse_container")
REPORT_BASE_PATH_WIDGET = dbutils.widgets.get("report_base_path").strip()
REPORT_BASE_PATH = REPORT_BASE_PATH_WIDGET or f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/external/hant-catalog/ge/reports"

GE_LAYER_CONFIGS = {
    "bronze": {
        "validated_table": "bronze_vehicle_positions_validated_stream",
        "quarantine_table": "bronze_vehicle_positions_ge_quarantine_stream",
        "results_table": "bronze_ge_results_hsl_vehicle_positions",
        "details_table": "bronze_ge_details_hsl_vehicle_positions",
        "samples_table": "bronze_ge_failed_samples_hsl_vehicle_positions",
        "rule_metrics_table": "bronze_ge_rule_metrics_hsl_vehicle_positions",
        "gate_table": "bronze_gate_results_hsl_vehicle_positions",
        "status_col": "validation_status",
        "rule_ids_col": "failed_rule_ids",
        "severities_col": "failed_severities",
        "descriptions_col": "failed_rule_descriptions",
        "rule_count_col": "failed_rule_count",
        "validation_ts_col": "validation_ts",
        "validation_date_col": "validation_date",
    },
    "silver": {
        "validated_table": "silver_vehicle_positions_validated_stream",
        "quarantine_table": "silver_vehicle_positions_ge_quarantine_stream",
        "results_table": "silver_ge_results_hsl_vehicle_positions",
        "details_table": "silver_ge_details_hsl_vehicle_positions",
        "samples_table": "silver_ge_failed_samples_hsl_vehicle_positions",
        "rule_metrics_table": "silver_ge_rule_metrics_hsl_vehicle_positions",
        "gate_table": "silver_gate_results_hsl_vehicle_positions",
        "status_col": "silver_validation_status",
        "rule_ids_col": "silver_failed_rule_ids",
        "severities_col": "silver_failed_severities",
        "descriptions_col": "silver_failed_rule_descriptions",
        "rule_count_col": "silver_failed_rule_count",
        "validation_ts_col": "silver_validation_ts",
        "validation_date_col": "silver_validation_date",
    },
    "gold": {
        "validated_table": "gold_vehicle_positions_validated_stream",
        "quarantine_table": "gold_vehicle_positions_ge_quarantine_stream",
        "results_table": "gold_ge_results_hsl_vehicle_positions",
        "details_table": "gold_ge_details_hsl_vehicle_positions",
        "samples_table": "gold_ge_failed_samples_hsl_vehicle_positions",
        "rule_metrics_table": "gold_ge_rule_metrics_hsl_vehicle_positions",
        "gate_table": "gold_gate_results_hsl_vehicle_positions",
        "status_col": "gold_validation_status",
        "rule_ids_col": "gold_failed_rule_ids",
        "severities_col": "gold_failed_severities",
        "descriptions_col": "gold_failed_rule_descriptions",
        "rule_count_col": "gold_failed_rule_count",
        "validation_ts_col": "gold_validation_ts",
        "validation_date_col": "gold_validation_date",
    },
}

REPORT_TABLES = {
    "summary": "report_ge_detection_summary",
    "decision_matrix": "report_ge_detection_matrix",
    "severity_source_type": "report_ge_detection_severity_source_type",
    "top_rules": "report_ge_detection_top_rules",
    "failed_records": "report_ge_detection_failed_records",
    "runtime": "report_ge_detection_runtime_summary",
}


def qname(table_name: str) -> str:
    return f"`{CATALOG_NAME}`.{SCHEMA_NAME}.{table_name}"


def table_exists(table_name: str) -> bool:
    try:
        spark.table(qname(table_name)).limit(1).collect()
        return True
    except Exception:
        return False


def save_report(df: DataFrame, table_name: str, partition_cols=None) -> None:
    if not PERSIST_REPORT_TABLES:
        return
    path = f"{REPORT_BASE_PATH.rstrip('/')}/{table_name}"
    writer = (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
    )
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.save(path)
    spark.sql(f"DROP TABLE IF EXISTS {qname(table_name)}")
    spark.sql(f"CREATE TABLE {qname(table_name)} USING DELTA LOCATION '{path}'")
    spark.sql(f"REFRESH TABLE {qname(table_name)}")
    print(json.dumps({"saved_table": qname(table_name), "path": path}, default=str))


def empty_array_string():
    return F.array().cast("array<string>")


def source_type_from_rule_id(rule_col):
    name = F.lower(rule_col)
    return (
        F.when(name.rlike("future|stale|timestamp|producer_ts|event_ts|time|delay|freshness"), F.lit("timeliness"))
         .when(name.rlike("parse|json|mqtt|schema|topic|format"), F.lit("schema_parse"))
         .when(name.rlike("null|missing|present|required"), F.lit("completeness"))
         .when(name.rlike("range|invalid|unexpected|bounds|negative|lat|lon|speed|occupancy|hour|domain"), F.lit("validity"))
         .when(name.rlike("duplicate|unique|mismatch|business_key|consistency"), F.lit("consistency"))
         .otherwise(F.lit("other"))
    )


def severity_rank(severity_col):
    s = F.lower(severity_col)
    return (
        F.when(s == "critical", F.lit(4))
         .when(s == "high", F.lit(3))
         .when(s == "warning", F.lit(3))
         .when(s == "medium", F.lit(2))
         .when(s == "low", F.lit(1))
         .otherwise(F.lit(0))
    )


def severity_from_rank(rank_col):
    return (
        F.when(rank_col >= 4, F.lit("critical"))
         .when(rank_col == 3, F.lit("high"))
         .when(rank_col == 2, F.lit("medium"))
         .when(rank_col == 1, F.lit("low"))
         .otherwise(F.lit("normal"))
    )


print(json.dumps({
    "ge_tables": {layer: {k: qname(v) for k, v in cfg.items() if k.endswith("_table")} for layer, cfg in GE_LAYER_CONFIGS.items()},
    "report_start_date": REPORT_START_DATE or None,
    "report_end_date": REPORT_END_DATE or None,
    "persist_report_tables": PERSIST_REPORT_TABLES,
    "report_base_path": REPORT_BASE_PATH,
}, indent=2))


{
  "ge_tables": {
    "bronze": {
      "validated_table": "`hant-catalog`.hsl.bronze_vehicle_positions_validated_stream",
      "quarantine_table": "`hant-catalog`.hsl.bronze_vehicle_positions_ge_quarantine_stream",
      "results_table": "`hant-catalog`.hsl.bronze_ge_results_hsl_vehicle_positions",
      "details_table": "`hant-catalog`.hsl.bronze_ge_details_hsl_vehicle_positions",
      "samples_table": "`hant-catalog`.hsl.bronze_ge_failed_samples_hsl_vehicle_positions",
      "rule_metrics_table": "`hant-catalog`.hsl.bronze_ge_rule_metrics_hsl_vehicle_positions",
      "gate_table": "`hant-catalog`.hsl.bronze_gate_results_hsl_vehicle_positions"
    },
    "silver": {
      "validated_table": "`hant-catalog`.hsl.silver_vehicle_positions_validated_stream",
      "quarantine_table": "`hant-catalog`.hsl.silver_vehicle_positions_ge_quarantine_stream",
      "results_table": "`hant-catalog`.hsl.silver_ge_results_hsl_vehicle_positions",
      "details_table": "`hant-catalog`.hsl.silver_g

In [0]:
def ge_record_id_expr(layer: str, cols: list[str]):
    if layer == "bronze" and {"topic", "partition", "offset"}.issubset(set(cols)):
        return F.concat_ws("|", F.coalesce(F.col("topic"), F.lit("")), F.col("partition").cast("string"), F.col("offset").cast("string"))
    if "business_key" in cols:
        return F.col("business_key").cast("string")
    if {"vehicle_id", "event_ts_unix", "route_id", "direction_id"}.issubset(set(cols)):
        return F.concat_ws("|", F.col("vehicle_id"), F.col("event_ts_unix").cast("string"), F.col("route_id"), F.col("direction_id"))
    return F.sha2(F.to_json(F.struct(*[F.col(c) for c in cols])), 256)


def load_ge_record_layer(layer: str, config: dict) -> DataFrame | None:
    frames = []
    missing = []

    for output_type, table_name in [
        ("validated", config["validated_table"]),
        ("quarantine", config["quarantine_table"]),
    ]:
        if not table_exists(table_name):
            missing.append(qname(table_name))
            continue

        df = spark.table(qname(table_name))
        cols = df.columns
        status_col = config["status_col"]
        rule_ids_col = config["rule_ids_col"]
        severities_col = config["severities_col"]
        descriptions_col = config["descriptions_col"]
        rule_count_col = config["rule_count_col"]
        validation_ts_col = config["validation_ts_col"]
        validation_date_col = config["validation_date_col"]

        rule_ids = F.col(rule_ids_col) if rule_ids_col in cols else empty_array_string()
        severities = F.col(severities_col) if severities_col in cols else empty_array_string()
        descriptions = F.col(descriptions_col) if descriptions_col in cols else empty_array_string()
        rule_count = F.col(rule_count_col).cast("long") if rule_count_col in cols else F.size(rule_ids).cast("long")
        status = F.col(status_col).cast("string") if status_col in cols else F.lit(output_type)
        validation_ts = F.col(validation_ts_col).cast("timestamp") if validation_ts_col in cols else F.current_timestamp()
        validation_date = F.to_date(F.col(validation_date_col)) if validation_date_col in cols else F.to_date(validation_ts)
        ingest_date = F.to_date("ingest_date") if "ingest_date" in cols else validation_date
        event_ts = F.col("event_ts").cast("timestamp") if "event_ts" in cols else F.lit(None).cast("timestamp")
        business_key = F.col("business_key").cast("string") if "business_key" in cols else F.lit(None).cast("string")

        frames.append(
            df.select(
                F.lit(layer).alias("layer"),
                F.lit(output_type).alias("ge_output_type"),
                ge_record_id_expr(layer, cols).alias("record_id"),
                business_key.alias("business_key"),
                event_ts.alias("event_ts"),
                ingest_date.alias("ingest_date"),
                validation_ts.alias("validation_ts"),
                validation_date.alias("validation_date"),
                status.alias("ge_status"),
                rule_ids.alias("failed_rule_ids"),
                severities.alias("failed_severities"),
                descriptions.alias("failed_rule_descriptions"),
                rule_count.alias("failed_rule_count"),
            )
        )

    if missing:
        print(f"Missing GE record tables for {layer}:")
        for table_name in missing:
            print(f"  - {table_name}")

    if not frames:
        return None

    layer_df = reduce(lambda left, right: left.unionByName(right, allowMissingColumns=True), frames)
    max_rank = F.array_max(F.transform(F.col("failed_severities"), lambda x: severity_rank(x)))

    return (
        layer_df
        .withColumn("ge_detected", F.coalesce(F.col("failed_rule_count"), F.lit(0)) > 0)
        .withColumn("ge_severity_rank", F.coalesce(max_rank, F.lit(0)))
        .withColumn("ge_severity", severity_from_rank(F.col("ge_severity_rank")))
        .withColumn(
            "ge_decision_action",
            F.when(F.col("ge_detected"), F.lit("QUARANTINE")).otherwise(F.lit("PASS"))
        )
    )


record_frames = [
    df for df in [
        load_ge_record_layer(layer, config)
        for layer, config in GE_LAYER_CONFIGS.items()
    ]
    if df is not None
]

if not record_frames:
    raise ValueError("No GE validated/quarantine record tables found.")

ge_records = reduce(lambda left, right: left.unionByName(right, allowMissingColumns=True), record_frames)

if REPORT_START_DATE:
    ge_records = ge_records.where(F.col("ingest_date") >= F.to_date(F.lit(REPORT_START_DATE)))
if REPORT_END_DATE:
    ge_records = ge_records.where(F.col("ingest_date") <= F.to_date(F.lit(REPORT_END_DATE)))

ge_records = ge_records.cache()

display(ge_records.groupBy("layer", "ingest_date").agg(
    F.count("*").alias("validated_records"),
    F.sum(F.when(F.col("ge_detected"), 1).otherwise(0)).cast("bigint").alias("detected_records"),
    F.min("validation_ts").alias("first_validation_ts"),
    F.max("validation_ts").alias("last_validation_ts"),
).orderBy("layer", "ingest_date"))


layer,ingest_date,validated_records,detected_records,first_validation_ts,last_validation_ts
bronze,2026-04-29,142385,142385,2026-04-29T05:29:58.695771Z,2026-04-29T05:29:58.695771Z
gold,2026-04-29,94174,0,2026-04-29T07:12:46.66286Z,2026-04-29T07:12:46.66286Z
silver,2026-04-29,128280,128280,2026-04-29T05:51:58.642139Z,2026-04-29T05:51:58.642139Z


In [0]:
summary_df = (
    ge_records
    .groupBy("layer", "ingest_date")
    .agg(
        F.count("*").alias("total_validated_records"),
        F.sum(F.when(~F.col("ge_detected"), 1).otherwise(0)).cast("bigint").alias("pass_records"),
        F.sum(F.when(F.col("ge_detected"), 1).otherwise(0)).cast("bigint").alias("quarantine_records"),
        F.round(F.avg(F.when(~F.col("ge_detected"), 1.0).otherwise(0.0)) * 100, 2).alias("pass_pct"),
        F.round(F.avg(F.when(F.col("ge_detected"), 1.0).otherwise(0.0)) * 100, 2).alias("quarantine_pct"),
        F.sum(F.coalesce(F.col("failed_rule_count"), F.lit(0))).cast("bigint").alias("total_failed_rule_hits"),
        F.min("validation_ts").alias("first_validation_ts"),
        F.max("validation_ts").alias("last_validation_ts"),
    )
    .orderBy("layer", "ingest_date")
)

display(summary_df)
save_report(summary_df, REPORT_TABLES["summary"], ["ingest_date", "layer"])


layer,ingest_date,total_validated_records,pass_records,quarantine_records,pass_pct,quarantine_pct,total_failed_rule_hits,first_validation_ts,last_validation_ts
bronze,2026-04-29,142385,0,142385,0.0,100.0,142385,2026-04-29T05:29:58.695771Z,2026-04-29T05:29:58.695771Z
gold,2026-04-29,94174,94174,0,100.0,0.0,0,2026-04-29T07:12:46.66286Z,2026-04-29T07:12:46.66286Z
silver,2026-04-29,128280,0,128280,0.0,100.0,163665,2026-04-29T05:51:58.642139Z,2026-04-29T05:51:58.642139Z


{"saved_table": "`hant-catalog`.hsl.report_ge_detection_summary", "path": "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/ge/reports/report_ge_detection_summary"}


In [0]:
decision_matrix_df = (
    ge_records
    .groupBy("layer", "ingest_date", "ge_decision_action", "ge_status", "ge_severity")
    .agg(
        F.count("*").alias("record_count"),
        F.sum(F.coalesce(F.col("failed_rule_count"), F.lit(0))).cast("bigint").alias("failed_rule_hits"),
    )
    .orderBy("layer", "ingest_date", "ge_decision_action", "ge_status", "ge_severity")
)

display(decision_matrix_df)
save_report(decision_matrix_df, REPORT_TABLES["decision_matrix"], ["ingest_date", "layer"])


layer,ingest_date,ge_decision_action,ge_status,ge_severity,record_count,failed_rule_hits
bronze,2026-04-29,QUARANTINE,validated,medium,142385,142385
gold,2026-04-29,PASS,validated,normal,94174,0
silver,2026-04-29,QUARANTINE,quarantined,high,1204,3687
silver,2026-04-29,QUARANTINE,quarantined,medium,32902,65804
silver,2026-04-29,QUARANTINE,validated,medium,94174,94174


{"saved_table": "`hant-catalog`.hsl.report_ge_detection_matrix", "path": "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/ge/reports/report_ge_detection_matrix"}


In [0]:
exploded_rules_df = (
    ge_records
    .where(F.col("ge_detected"))
    .select(
        "layer",
        "ingest_date",
        "record_id",
        "business_key",
        F.posexplode_outer("failed_rule_ids").alias("rule_pos", "rule_id"),
        "failed_severities",
        "failed_rule_descriptions",
    )
    .where(F.col("rule_id").isNotNull())
    .withColumn("severity", F.coalesce(F.expr("try_element_at(failed_severities, rule_pos + 1)"), F.lit("unknown")))
    .withColumn("rule_description", F.expr("try_element_at(failed_rule_descriptions, rule_pos + 1)"))
    .withColumn("source_type", source_type_from_rule_id(F.col("rule_id")))
)

severity_source_type_df = (
    exploded_rules_df
    .groupBy("layer", "ingest_date", "source_type", "severity")
    .agg(
        F.count("*").alias("rule_hit_count"),
        F.countDistinct("record_id").alias("affected_records"),
    )
    .orderBy("layer", "ingest_date", F.desc("affected_records"), "source_type", "severity")
)

display(severity_source_type_df)
save_report(severity_source_type_df, REPORT_TABLES["severity_source_type"], ["ingest_date", "layer"])


layer,ingest_date,source_type,severity,rule_hit_count,affected_records
bronze,2026-04-29,timeliness,medium,142385,142385
silver,2026-04-29,timeliness,medium,128280,128280
silver,2026-04-29,schema_parse,low,32902,32902
silver,2026-04-29,validity,unknown,1204,1204
silver,2026-04-29,validity,high,1129,1129
silver,2026-04-29,schema_parse,high,75,75
silver,2026-04-29,validity,low,75,75


{"saved_table": "`hant-catalog`.hsl.report_ge_detection_severity_source_type", "path": "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/ge/reports/report_ge_detection_severity_source_type"}


In [0]:
warning_quarantine_df = (
    ge_records
    .where(F.col("ge_detected"))
    .select(
        "layer",
        "ingest_date",
        "record_id",
        "business_key",
        "event_ts",
        "ge_decision_action",
        "ge_status",
        "ge_severity",
        "failed_rule_count",
        "failed_rule_ids",
        "failed_severities",
        "failed_rule_descriptions",
        "validation_ts",
        "validation_date",
    )
    .orderBy("layer", "ingest_date", F.desc("failed_rule_count"))
)

display(warning_quarantine_df)
save_report(warning_quarantine_df, REPORT_TABLES["failed_records"], ["ingest_date", "layer"])

layer,ingest_date,record_id,business_key,event_ts,ge_decision_action,ge_status,ge_severity,failed_rule_count,failed_rule_ids,failed_severities,failed_rule_descriptions,validation_ts,validation_date
bronze,2026-04-29,vehicle-position-events|0|3623115,null,null,QUARANTINE,validated,medium,1,List(medium_producer_ts_bad_format),List(medium),List(Producer ingest timestamp should be ISO-8601 UTC ending with Z.),2026-04-29T05:29:58.695771Z,2026-04-29
bronze,2026-04-29,vehicle-position-events|0|3623108,null,null,QUARANTINE,validated,medium,1,List(medium_producer_ts_bad_format),List(medium),List(Producer ingest timestamp should be ISO-8601 UTC ending with Z.),2026-04-29T05:29:58.695771Z,2026-04-29
bronze,2026-04-29,vehicle-position-events|0|3623277,null,null,QUARANTINE,validated,medium,1,List(medium_producer_ts_bad_format),List(medium),List(Producer ingest timestamp should be ISO-8601 UTC ending with Z.),2026-04-29T05:29:58.695771Z,2026-04-29
bronze,2026-04-29,vehicle-position-events|0|3623322,null,null,QUARANTINE,validated,medium,1,List(medium_producer_ts_bad_format),List(medium),List(Producer ingest timestamp should be ISO-8601 UTC ending with Z.),2026-04-29T05:29:58.695771Z,2026-04-29
bronze,2026-04-29,vehicle-position-events|0|3623218,null,null,QUARANTINE,validated,medium,1,List(medium_producer_ts_bad_format),List(medium),List(Producer ingest timestamp should be ISO-8601 UTC ending with Z.),2026-04-29T05:29:58.695771Z,2026-04-29
bronze,2026-04-29,vehicle-position-events|0|3623384,null,null,QUARANTINE,validated,medium,1,List(medium_producer_ts_bad_format),List(medium),List(Producer ingest timestamp should be ISO-8601 UTC ending with Z.),2026-04-29T05:29:58.695771Z,2026-04-29
bronze,2026-04-29,vehicle-position-events|0|3623295,null,null,QUARANTINE,validated,medium,1,List(medium_producer_ts_bad_format),List(medium),List(Producer ingest timestamp should be ISO-8601 UTC ending with Z.),2026-04-29T05:29:58.695771Z,2026-04-29
bronze,2026-04-29,vehicle-position-events|0|3623392,null,null,QUARANTINE,validated,medium,1,List(medium_producer_ts_bad_format),List(medium),List(Producer ingest timestamp should be ISO-8601 UTC ending with Z.),2026-04-29T05:29:58.695771Z,2026-04-29
bronze,2026-04-29,vehicle-position-events|0|3623131,null,null,QUARANTINE,validated,medium,1,List(medium_producer_ts_bad_format),List(medium),List(Producer ingest timestamp should be ISO-8601 UTC ending with Z.),2026-04-29T05:29:58.695771Z,2026-04-29
bronze,2026-04-29,vehicle-position-events|0|3623620,null,null,QUARANTINE,validated,medium,1,List(medium_producer_ts_bad_format),List(medium),List(Producer ingest timestamp should be ISO-8601 UTC ending with Z.),2026-04-29T05:29:58.695771Z,2026-04-29


{"saved_table": "`hant-catalog`.hsl.report_ge_detection_failed_records", "path": "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/ge/reports/report_ge_detection_failed_records"}


In [0]:
top_rules_df = (
    exploded_rules_df
    .groupBy("layer", "ingest_date", "source_type", "severity", "rule_id", "rule_description")
    .agg(
        F.count("*").alias("rule_hit_count"),
        F.countDistinct("record_id").alias("affected_records"),
    )
    .orderBy("layer", "ingest_date", F.desc("affected_records"), F.desc("rule_hit_count"), "rule_id")
)

display(top_rules_df)
save_report(top_rules_df, REPORT_TABLES["top_rules"], ["ingest_date", "layer"])


layer,ingest_date,source_type,severity,rule_id,rule_description,rule_hit_count,affected_records
bronze,2026-04-29,timeliness,medium,medium_producer_ts_bad_format,Producer ingest timestamp should be ISO-8601 UTC ending with Z.,142385,142385
silver,2026-04-29,timeliness,medium,medium_event_stale_gt_5m,Event timestamp is more than 5 minutes older than Silver ingest time.,127076,127076
silver,2026-04-29,schema_parse,low,low_topic_operator_mismatch,Topic operator ID and payload operator ID should usually agree after normalization.,32902,32902
silver,2026-04-29,validity,unknown,high_longitude_out_of_bounds,Event timestamp is more than 5 minutes older than Silver ingest time.,1204,1204
silver,2026-04-29,timeliness,medium,medium_event_stale_gt_5m,Latitude must fall within HSL operating bounds.,1204,1204
silver,2026-04-29,validity,high,high_latitude_out_of_bounds,Longitude must fall within HSL operating bounds.,1129,1129
silver,2026-04-29,validity,low,high_latitude_out_of_bounds,Topic operator ID and payload operator ID should usually agree after normalization.,75,75
silver,2026-04-29,schema_parse,high,low_topic_operator_mismatch,Longitude must fall within HSL operating bounds.,75,75


{"saved_table": "`hant-catalog`.hsl.report_ge_detection_top_rules", "path": "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/ge/reports/report_ge_detection_top_rules"}


In [0]:
def load_gate_layer(layer: str, table_name: str) -> DataFrame | None:
    if not table_exists(table_name):
        print(f"Missing GE gate table: {qname(table_name)}")
        return None
    df = spark.table(qname(table_name)).withColumn("layer", F.lit(layer))
    cols = df.columns
    run_ts = F.col("run_ts_utc").cast("timestamp") if "run_ts_utc" in cols else F.current_timestamp()
    validation_date = F.to_date(F.col("validation_date")) if "validation_date" in cols else F.to_date(run_ts)
    select_cols = [
        F.lit(layer).alias("layer"),
        run_ts.alias("run_ts_utc"),
        validation_date.alias("validation_date"),
        F.col("gate_status").cast("string").alias("gate_status") if "gate_status" in cols else F.lit(None).cast("string").alias("gate_status"),
        F.col("input_row_count").cast("long").alias("input_row_count") if "input_row_count" in cols else F.lit(None).cast("long").alias("input_row_count"),
        F.col("validated_row_count").cast("long").alias("validated_row_count") if "validated_row_count" in cols else F.lit(None).cast("long").alias("validated_row_count"),
        F.col("quarantined_row_count").cast("long").alias("quarantined_row_count") if "quarantined_row_count" in cols else F.lit(None).cast("long").alias("quarantined_row_count"),
        F.col("quarantine_rate").cast("double").alias("quarantine_rate") if "quarantine_rate" in cols else F.lit(None).cast("double").alias("quarantine_rate"),
        F.col("critical_failed_rows").cast("long").alias("critical_failed_rows") if "critical_failed_rows" in cols else F.lit(None).cast("long").alias("critical_failed_rows"),
        F.col("high_failed_rows").cast("long").alias("high_failed_rows") if "high_failed_rows" in cols else F.lit(None).cast("long").alias("high_failed_rows"),
        F.col("medium_failed_rows").cast("long").alias("medium_failed_rows") if "medium_failed_rows" in cols else F.lit(None).cast("long").alias("medium_failed_rows"),
        F.col("low_failed_rows").cast("long").alias("low_failed_rows") if "low_failed_rows" in cols else F.lit(None).cast("long").alias("low_failed_rows"),
    ]
    return df.select(*select_cols)


gate_frames = [
    df for df in [
        load_gate_layer(layer, config["gate_table"])
        for layer, config in GE_LAYER_CONFIGS.items()
    ]
    if df is not None
]

if gate_frames:
    gate_df = reduce(lambda left, right: left.unionByName(right, allowMissingColumns=True), gate_frames)
    if REPORT_START_DATE:
        gate_df = gate_df.where(F.col("validation_date") >= F.to_date(F.lit(REPORT_START_DATE)))
    if REPORT_END_DATE:
        gate_df = gate_df.where(F.col("validation_date") <= F.to_date(F.lit(REPORT_END_DATE)))

    runtime_summary_df = (
        gate_df
        .groupBy("layer", "validation_date")
        .agg(
            F.count("*").alias("validation_batches"),
            F.sum("input_row_count").cast("bigint").alias("input_records"),
            F.sum("validated_row_count").cast("bigint").alias("validated_records"),
            F.sum("quarantined_row_count").cast("bigint").alias("quarantined_records"),
            F.round(F.avg("quarantine_rate") * 100, 2).alias("avg_quarantine_pct"),
            F.sum("critical_failed_rows").cast("bigint").alias("critical_failed_rows"),
            F.sum("high_failed_rows").cast("bigint").alias("high_failed_rows"),
            F.sum("medium_failed_rows").cast("bigint").alias("medium_failed_rows"),
            F.sum("low_failed_rows").cast("bigint").alias("low_failed_rows"),
            F.sum(F.when(F.col("gate_status") == "BLOCK", 1).otherwise(0)).cast("bigint").alias("block_batches"),
            F.sum(F.when(F.col("gate_status") == "WARN", 1).otherwise(0)).cast("bigint").alias("warn_batches"),
            F.sum(F.when(F.col("gate_status") == "PASS", 1).otherwise(0)).cast("bigint").alias("pass_batches"),
            F.min("run_ts_utc").alias("first_validation_run_ts"),
            F.max("run_ts_utc").alias("last_validation_run_ts"),
        )
        .orderBy("layer", "validation_date")
    )
    display(runtime_summary_df)
    save_report(runtime_summary_df, REPORT_TABLES["runtime"], ["validation_date", "layer"])
else:
    print("No GE gate result tables found.")


layer,validation_date,validation_batches,input_records,validated_records,quarantined_records,avg_quarantine_pct,critical_failed_rows,high_failed_rows,medium_failed_rows,low_failed_rows,block_batches,warn_batches,pass_batches,first_validation_run_ts,last_validation_run_ts
bronze,2026-04-29,1,142385,142385,0,0.0,0,0,142385,0,0,1,0,2026-04-29T05:29:58.695771Z,2026-04-29T05:29:58.695771Z
gold,2026-04-29,1,94174,94174,0,0.0,0,0,0,0,0,0,1,2026-04-29T07:12:46.66286Z,2026-04-29T07:12:46.66286Z
silver,2026-04-29,1,128280,94174,34106,26.59,0,1204,128280,32977,1,0,0,2026-04-29T05:51:58.642139Z,2026-04-29T05:51:58.642139Z


{"saved_table": "`hant-catalog`.hsl.report_ge_detection_runtime_summary", "path": "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/ge/reports/report_ge_detection_runtime_summary"}


In [0]:
print(f"GE report window: ingest_date {REPORT_START_DATE} to {REPORT_END_DATE}")
summary_rows = summary_df.collect()
for row in summary_rows:
    print(
        f"{row['layer'].upper()} {row['ingest_date']}: "
        f"total={row['total_validated_records']}, "
        f"PASS={row['pass_records']} ({row['pass_pct']}%), "
        f"QUARANTINE={row['quarantine_records']} ({row['quarantine_pct']}%), "
        f"failed_rule_hits={row['total_failed_rule_hits']}"
    )
print("Persisted GE report tables:" if PERSIST_REPORT_TABLES else "Generated GE report DataFrames without persisting.")
for table_name in REPORT_TABLES.values():
    print(f"- {qname(table_name)}")

GE report window: ingest_date 2026-04-29 to 2026-04-29
BRONZE 2026-04-29: total=142385, PASS=0 (0.0%), QUARANTINE=142385 (100.0%), failed_rule_hits=142385
GOLD 2026-04-29: total=94174, PASS=94174 (100.0%), QUARANTINE=0 (0.0%), failed_rule_hits=0
SILVER 2026-04-29: total=128280, PASS=0 (0.0%), QUARANTINE=128280 (100.0%), failed_rule_hits=163665
Persisted GE report tables:
- `hant-catalog`.hsl.report_ge_detection_summary
- `hant-catalog`.hsl.report_ge_detection_matrix
- `hant-catalog`.hsl.report_ge_detection_severity_source_type
- `hant-catalog`.hsl.report_ge_detection_top_rules
- `hant-catalog`.hsl.report_ge_detection_failed_records
- `hant-catalog`.hsl.report_ge_detection_runtime_summary
